In [1]:
import cv2
import json
import os
import torch
from ultralytics.models import YOLO

# --- CONFIG ---
VIDEO_FOLDER = "D:/GitHub/BaseballPitch/modules/pitcher_segmentation/finetuning_dataset/videos"
OUTPUT_BASE = "D:/GitHub/BaseballPitch/modules/pitcher_segmentation/finetuning_dataset"
SPLIT = "train"  # 'train' or 'val' or 'test'
PITCH_TYPE = "CH - Changeup"
POSE_MODEL = 'D:/GitHub/BaseballPitch/yolo26n-pose.pt'  # Detect people with pose keypoints
CONF = 0.25  # Confidence threshold for pose detection
IMG_SIZE = 1280
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Frame skip: only label every Nth frame to speed up (1 = every frame)
FRAME_SKIP = 1  # Label every Nth frame
MAX_VIDEOS = None  # Set to a number to limit videos, or None for all
# Progress tracking file — records which videos are fully labeled
PROGRESS_FILE = f"{OUTPUT_BASE}/labeling_progress.json"
# --------------

In [2]:
class SemiAutoLabeler:
    def __init__(self):
        self.pose_model = YOLO(POSE_MODEL)
        self.pose_model.to(DEVICE)
        self.current_frame = None
        self.current_frame_num = 0
        self.video_name = ""
        self.labeled_count = 0
        self.skipped_count = 0
        self.deleted_count = 0
        self.detected_boxes = []
        self.detected_keypoints = []
        self.mouse_x = 0
        self.mouse_y = 0
        self.progress = self._load_progress()

    # ── Progress tracking ──────────────────────────────────────

    def _load_progress(self):
        if os.path.exists(PROGRESS_FILE):
            with open(PROGRESS_FILE, "r") as f:
                return json.load(f)
        return {}

    def _save_progress(self):
        os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
        with open(PROGRESS_FILE, "w") as f:
            json.dump(self.progress, f, indent=2)

    def _progress_key(self, video_name):
        return f"{SPLIT}/{PITCH_TYPE}/{video_name}"

    # ── Mouse callback ─────────────────────────────────────────

    def mouse_callback(self, event, x, y, flags, param):
        self.mouse_x = x
        self.mouse_y = y
        if event == cv2.EVENT_LBUTTONDOWN:
            for idx, box in enumerate(self.detected_boxes):
                x1, y1, x2, y2 = box
                if x1 <= x <= x2 and y1 <= y <= y2:
                    self.save_label(idx)
                    break

    # ── Save helpers ───────────────────────────────────────────

    def save_label(self, detection_idx):
        """Save YOLO-pose format label (box + 17 keypoints) for selected detection."""
        if self.current_frame is None:
            return

        h, w = self.current_frame.shape[:2]
        x1, y1, x2, y2 = self.detected_boxes[detection_idx]
        kpts = self.detected_keypoints[detection_idx]

        # Normalised box
        x_center = ((x1 + x2) / 2) / w
        y_center = ((y1 + y2) / 2) / h
        box_width = (x2 - x1) / w
        box_height = (y2 - y1) / h

        # Keypoints: x y v  (v=2 visible, v=0 not visible)
        kpt_parts = []
        for kx, ky, kconf in kpts:
            if kconf > 0.5:
                kpt_parts.extend([f"{kx / w:.6f}", f"{ky / h:.6f}", "2"])
            else:
                kpt_parts.extend(["0.000000", "0.000000", "0"])

        # Save image
        img_name = f"{self.video_name}_{self.current_frame_num:04d}.jpg"
        img_path = f"{OUTPUT_BASE}/images/{SPLIT}/{PITCH_TYPE}/{img_name}"
        os.makedirs(os.path.dirname(img_path), exist_ok=True)
        cv2.imwrite(img_path, self.current_frame)

        # Save label
        label_path = (
            f"{OUTPUT_BASE}/labels/{SPLIT}/{PITCH_TYPE}/"
            f"{self.video_name}_{self.current_frame_num:04d}.txt"
        )
        os.makedirs(os.path.dirname(label_path), exist_ok=True)
        with open(label_path, "w") as f:
            f.write(
                f"0 {x_center:.6f} {y_center:.6f} "
                f"{box_width:.6f} {box_height:.6f} "
                f"{' '.join(kpt_parts)}\n"
            )

        self.labeled_count += 1
        print(f"✓ Labeled frame {self.current_frame_num} ({self.labeled_count} total)")

    def save_frame_no_label(self):
        """Save frame as a negative example (image + empty label file)."""
        if self.current_frame is None:
            return

        img_name = f"{self.video_name}_{self.current_frame_num:04d}.jpg"
        img_path = f"{OUTPUT_BASE}/images/{SPLIT}/{PITCH_TYPE}/{img_name}"
        os.makedirs(os.path.dirname(img_path), exist_ok=True)
        cv2.imwrite(img_path, self.current_frame)

        label_path = (
            f"{OUTPUT_BASE}/labels/{SPLIT}/{PITCH_TYPE}/"
            f"{self.video_name}_{self.current_frame_num:04d}.txt"
        )
        os.makedirs(os.path.dirname(label_path), exist_ok=True)
        open(label_path, "w").close()

        self.skipped_count += 1
        print(f"⏭️  Saved frame {self.current_frame_num} without label ({self.skipped_count} total)")

    # ── Per-video labeling loop ────────────────────────────────

    def label_video(self, video_path):
        self.video_name = os.path.splitext(os.path.basename(video_path))[0]
        key = self._progress_key(self.video_name)

        # Fully done → skip entirely
        if key in self.progress and self.progress[key].get("status") == "done":
            print(f"⏭️  Skipping {self.video_name} (already completed)")
            return True

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f"\n{'='*60}")
        print(f"Video: {self.video_name}  |  Split: {SPLIT}")
        print(f"Total frames: {total_frames} (every {FRAME_SKIP})")
        print(f"{'='*60}")
        print("Controls:  Click/A = select pitcher  |  SPACE = no label  |  D = delete  |  N = next video  |  Q = quit")

        cv2.namedWindow("Semi-Auto Labeling")
        cv2.setMouseCallback("Semi-Auto Labeling", self.mouse_callback)

        frames_to_label = list(range(0, total_frames, FRAME_SKIP))

        # Partially labeled → find remaining frames
        label_dir = f"{OUTPUT_BASE}/labels/{SPLIT}/{PITCH_TYPE}"
        already_labeled = set()
        if os.path.exists(label_dir):
            for fname in os.listdir(label_dir):
                if fname.startswith(self.video_name + "_") and fname.endswith(".txt"):
                    stem = os.path.splitext(fname)[0]
                    parts = stem.rsplit("_", 1)
                    if len(parts) == 2 and parts[1].isdigit():
                        already_labeled.add(int(parts[1]))

        remaining = [f for f in frames_to_label if f not in already_labeled]
        if already_labeled:
            print(f"Resuming: {len(already_labeled)} done, {len(remaining)} remaining")

        for ri, frame_idx in enumerate(remaining):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break

            self.current_frame = frame.copy()
            self.current_frame_num = frame_idx
            self.detected_boxes = []
            self.detected_keypoints = []

            # Run pose model
            pose_results = self.pose_model(
                frame, conf=CONF, imgsz=IMG_SIZE, device=DEVICE, verbose=False
            )
            result = pose_results[0]
            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxy
                if isinstance(boxes, torch.Tensor):
                    boxes = boxes.cpu().numpy()
                kpts_data = result.keypoints.data
                if isinstance(kpts_data, torch.Tensor):
                    kpts_data = kpts_data.cpu().numpy()

                for i, box in enumerate(boxes):
                    x1, y1, x2, y2 = map(int, box[:4])
                    self.detected_boxes.append((x1, y1, x2, y2))
                    self.detected_keypoints.append(kpts_data[i])

            # Interactive loop for this frame
            while True:
                display = self.current_frame.copy()

                for i, (bx1, by1, bx2, by2) in enumerate(self.detected_boxes):
                    cv2.rectangle(display, (bx1, by1), (bx2, by2), (0, 255, 0), 2)
                    cv2.putText(display, f"Person {i+1}", (bx1, by1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                info = f"Frame {frame_idx}/{total_frames} | Progress: {ri+1}/{len(remaining)}"
                cv2.putText(display, info, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(display, info, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 1)

                hint = ("Click pitcher or 'A' | SPACE=no label | D=delete"
                        if self.detected_boxes
                        else "No detections | SPACE=no label | D=delete")
                cv2.putText(display, hint, (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(display, hint, (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

                cv2.imshow("Semi-Auto Labeling", display)
                key = cv2.waitKeyEx(1)

                if key in (ord("a"), ord("A")):
                    if self.detected_boxes:
                        selected_idx = None
                        for idx, (bx1, by1, bx2, by2) in enumerate(self.detected_boxes):
                            if bx1 <= self.mouse_x <= bx2 and by1 <= self.mouse_y <= by2:
                                selected_idx = idx
                                break
                        if selected_idx is not None:
                            self.save_label(selected_idx)
                            break
                        else:
                            print("⚠️  Move mouse over a bounding box to select with 'A'")
                    else:
                        print("⚠️  No bounding boxes detected")

                elif key in (32, 2555904, ord("s"), ord("S")):
                    self.save_frame_no_label()
                    break

                elif key in (ord("d"), ord("D")):
                    self.deleted_count += 1
                    print(f"🗑️  Deleted frame {frame_idx} ({self.deleted_count} total)")
                    break

                elif key in (ord("q"), ord("Q"), 27):
                    cap.release()
                    cv2.destroyAllWindows()
                    return False

                elif key in (ord("n"), ord("N")):
                    cap.release()
                    return True

                # Also break if mouse-click already saved this frame
                img_check = (
                    f"{OUTPUT_BASE}/images/{SPLIT}/{PITCH_TYPE}/"
                    f"{self.video_name}_{self.current_frame_num:04d}.jpg"
                )
                if os.path.exists(img_check):
                    break

        cap.release()

        # Mark video done
        self.progress[key] = {
            "status": "done",
            "labeled": self.labeled_count,
            "skipped": self.skipped_count,
            "deleted": self.deleted_count,
        }
        self._save_progress()
        return True

    # ── Main entry point ───────────────────────────────────────

    def run(self):
        video_dir = f"{VIDEO_FOLDER}/{SPLIT}/{PITCH_TYPE}"
        videos = sorted([f for f in os.listdir(video_dir) if f.endswith(".mp4")])

        if MAX_VIDEOS is not None:
            videos = videos[:MAX_VIDEOS]

        if not videos:
            print(f"No videos found in {video_dir}")
            return

        done_count = sum(
            1 for v in videos
            if self.progress.get(
                self._progress_key(os.path.splitext(v)[0]), {}
            ).get("status") == "done"
        )
        print(f"Found {len(videos)} videos ({done_count} already completed)")
        print("Starting semi-automatic labeling session...\n")

        for i, video in enumerate(videos, 1):
            print(f"[Video {i}/{len(videos)}]")
            if not self.label_video(os.path.join(video_dir, video)):
                break

        cv2.destroyAllWindows()
        print(f"\n{'='*60}")
        print("Labeling session complete!")
        print(f"  Labeled (with pitcher): {self.labeled_count}")
        print(f"  Saved (no pitcher):     {self.skipped_count}")
        print(f"  Deleted:                {self.deleted_count}")
        print("="*60)

In [6]:
labeler = SemiAutoLabeler()
labeler.run()

Found 21 videos (20 already completed)

Starting semi-automatic labeling session...

[Video 1/21]
⏭️  Skipping PitchType-CH_Zone-11_PlayID-cd61c6db-63f7-341d-9928-517e2778d741_Date-2025-09-02 (already completed in progress log)

[Video 2/21]
⏭️  Skipping PitchType-CH_Zone-11_PlayID-e5658640-5139-3bcc-b490-4ba9332bde5c_Date-2025-09-02 (already completed in progress log)

[Video 3/21]
⏭️  Skipping PitchType-CH_Zone-11_PlayID-f1529eed-27cf-3798-beef-64ea0eaca713_Date-2025-09-13 (already completed in progress log)

[Video 4/21]
⏭️  Skipping PitchType-CH_Zone-13_PlayID-3b4a2254-1988-3053-929c-cf058e047375_Date-2025-08-22 (already completed in progress log)

[Video 5/21]
⏭️  Skipping PitchType-CH_Zone-13_PlayID-9a15fc57-fcd9-3050-967f-1f4ffa875792_Date-2025-08-22 (already completed in progress log)

[Video 6/21]
⏭️  Skipping PitchType-CH_Zone-13_PlayID-9ed1178f-cb29-3749-9c22-dc58cea7d176_Date-2025-08-22 (already completed in progress log)

[Video 7/21]
⏭️  Skipping PitchType-CH_Zone-13_Play